# Week 13: Reinforcement Learning & RLHF
## Notebook 06: Proximal Policy Optimization (PPO) Clipped Objective from Scratch

### 1. Conceptual Intuition & Real-World Use Case
Standard Policy Gradient algorithms (REINFORCE) suffer from destructive policy updates:
- A single bad mini-batch with high variance gradients can destroy policy parameters, causing performance to collapse permanently.
- Natural Policy Gradients and TRPO (Trust Region Policy Optimization) enforce mathematical trust regions via second-order Hessian constraints, but are computationally expensive.

In 2017, Schulman et al. (OpenAI) released **PPO (Proximal Policy Optimization)**:
- Employs a first-order **Clipped Surrogate Objective**.
- Clips the probability ratio $r_t(\theta)$ within $[1-\epsilon, 1+\epsilon]$ whenever the update moves too far from the old policy.
- **The Core Algorithm Behind RLHF in ChatGPT!**

---

### 2. Mathematical Foundations: PPO Clipped Objective
Let the probability ratio between the new policy $\pi_\theta$ and the old policy $\pi_{\theta_{old}}$ be:
$$r_t(\theta) = \frac{\pi_\theta(a_t \mid s_t)}{\pi_{\theta_{old}}(a_t \mid s_t)}$$

The clipped surrogate objective is:
$$L^{\text{CLIP}}(\theta) = \hat{\mathbb{E}}_t \left[ \min\left( r_t(\theta) A_t, \; \text{clip}(r_t(\theta), 1 - \epsilon, 1 + \epsilon) A_t \right) \right]$$

Where:
- $A_t$ is the Generalized Advantage Estimate (GAE).
- $\epsilon \approx 0.2$ is the clipping boundary.
- **Pessimistic Bound:** Taking the minimum acts as a lower bound, ensuring the policy does not claim credit for policy changes beyond the trust region!

In [ ]:
import numpy as np

# Step 1: Simulate Advantage and Probability Ratios
epsilon = 0.2  # Clipping parameter (allows 20% shift)

# Generate an array of policy ratios r_t(theta) from 0.5 to 1.5
ratios = np.linspace(0.5, 1.5, 100)

def compute_ppo_surrogate(r, advantage, eps=0.2):
    surr1 = r * advantage
    surr2 = np.clip(r, 1.0 - eps, 1.0 + eps) * advantage
    return np.minimum(surr1, surr2)

# Case A: Positive Advantage (A = +1.0) -> Action was better than average!
# Policy wants to increase r, but clipping caps the reward at 1 + epsilon
ppo_pos = compute_ppo_surrogate(ratios, advantage=1.0, eps=epsilon)

# Case B: Negative Advantage (A = -1.0) -> Action was worse than average!
# Policy wants to decrease r, but clipping caps the penalty at 1 - epsilon
ppo_neg = compute_ppo_surrogate(ratios, advantage=-1.0, eps=epsilon)

print("--- PPO Clipped Surrogate Loss Behavior ---")
print(f"When Advantage A = +1.0 and Ratio r = 1.40 (40% increase):")
print(f"  Unclipped Objective (r * A):       1.40")
print(f"  Clipped PPO Objective:             {compute_ppo_surrogate(1.40, 1.0):.2f} (Capped at 1.20!)")

print(f"\nWhen Advantage A = -1.0 and Ratio r = 0.60 (40% decrease):")
print(f"  Unclipped Objective (r * A):       -0.60")
print(f"  Clipped PPO Objective:             {compute_ppo_surrogate(0.60, -1.0):.2f} (Bounded!)")

### 3. Step-by-Step Code Walkthrough
1. **Ratio Definition:** $r=1.0$ corresponds to unchanged policy.
2. **Clipping Safeguard:** When an action had positive advantage ($A>0$), the optimizer is discouraged from increasing its probability beyond $1+\epsilon=1.20$.
3. **Pessimistic Min:** Prevents aggressive step sizes, guaranteeing stable continuous policy updates over hundreds of training epochs.

---

### 4. Industry Applications & Trade-offs
- **ChatGPT Alignment:** PPO aligns the LLM policy against a human preference reward model while using a KL-penalty to prevent policy collapse.
- **Robotics:** Boston Dynamics and OpenAI gym locomotion controllers.

---

### 5. Top Interview Questions & Answers
**Q1: Why take the $\min()$ of the unclipped and clipped objectives in PPO?**
> **Answer:** The minimum creates a pessimistic lower bound. If the unclipped objective produces an overly optimistic huge step, the minimum truncates it to the clipped value, preventing destabilizing updates.

**Q2: What happens if an action has negative advantage ($A < 0$) and the policy ratio $r > 1+\epsilon$?**
> **Answer:** If $A < 0$ and $r > 1.2$, the action became *more* likely despite being bad! In this case, $\min(r A, \text{clip} A) = r A$. The clipping does not apply, allowing full gradient penalty to push the probability down rapidly.